# transformers.js #1599 — where the Qwen3.5-4B prefill time goes

**What this notebook establishes:** the exported ONNX graph for `onnx-community/Qwen3.5-4B-ONNX`
evaluates its 24 gated-DeltaNet layers with a **sequential ONNX `Scan` over the sequence axis**
on the prefill path. Not a shader problem — a graph-structure problem.

**Why it's cheap to check:** the ONNX weights live in sibling `.onnx_data` files, so the *graph*
is only ~1.4 MB. No GPU, no auth, no 2.5 GB download. Runs in well under a minute on a free CPU runtime.

**What this notebook does _not_ claim.** It contains no timing. transformers.js runs WebGPU in a
browser; Colab is a Python VM with no browser and no WebGPU, so wall-clock numbers measured here
would not transfer. Everything below is a *static property of the published file* — deterministic,
and identical for anyone who runs it.

In [ ]:
%pip install -q onnx huggingface_hub

In [ ]:
from huggingface_hub import hf_hub_download

REPO = "onnx-community/Qwen3.5-4B-ONNX"
graph_path  = hf_hub_download(REPO, "onnx/decoder_model_merged_q4f16.onnx")  # ~1.4 MB, graph only
config_path = hf_hub_download(REPO, "config.json")
print(graph_path)

## 1. Op census — walking every subgraph

Ops hide inside `If`/`Scan` bodies, so a flat pass over `model.graph.node` undercounts. This recurses.

In [ ]:
import onnx, collections

model = onnx.load(graph_path, load_external_data=False)   # graph only; weights not needed

ops = collections.Counter()
def walk(g):
    for n in g.node:
        ops[n.op_type] += 1
        for a in n.attribute:
            if a.g and a.g.node:
                walk(a.g)
walk(model.graph)

print(f"Scan                          : {ops['Scan']:>3}   <- sequential recurrence")
print(f"GroupQueryAttention           : {ops['GroupQueryAttention']:>3}")
print(f"LinearAttention (ORT fused op) : {ops['LinearAttention']:>3}   <- absent from this graph")

## 2. Are those `Scan`s on the prefill path?

Each gated-DeltaNet layer emits an `If` switching on `/model/layers.N/gdn/is_decode`.
If every `Scan` sits in the **`else_branch`** (the not-decode side), the sequential recurrence
is specifically what a *prompt* pays.

In [ ]:
hits = []
def hunt(g):
    for n in g.node:
        if n.op_type == "If":
            cond = n.input[0] if n.input else "?"
            for a in n.attribute:
                if a.g and any(x.op_type == "Scan" for x in a.g.node):
                    hits.append((cond, a.name))
        for a in n.attribute:
            if a.g and a.g.node:
                hunt(a.g)
hunt(model.graph)

branches = collections.Counter(b for _, b in hits)
print(f"If-branches directly containing a Scan: {len(hits)}")
print(f"which branch: {dict(branches)}")
print("conditions (first 4):")
for c, b in hits[:4]:
    print(f"   {c}  ->  {b}")

## 3. What one `Scan` actually does

Its body is the gated delta rule; the scan axes say which dimension it walks.

In [ ]:
def first_scan(g):
    for n in g.node:
        if n.op_type == "Scan":
            return n
        for a in n.attribute:
            if a.g and a.g.node:
                r = first_scan(a.g)
                if r is not None:
                    return r
    return None

scan = first_scan(model.graph)
body = [a.g for a in scan.attribute if a.g][0]
axes = [list(a.ints) for a in scan.attribute if a.name == "scan_input_axes"]

print("body inputs      :", [i.name for i in body.input])
print("body node count  :", len(body.node))
print("body ops         :", collections.Counter(n.op_type for n in body.node).most_common())
print("scan_input_axes  :", axes, " <- axis 2 = sequence: one position per iteration")

## 4. Cross-check against `config.json`

The architecture should agree with the op census: 24 linear-attention layers, 8 full-attention.

In [ ]:
import json
cfg = json.load(open(config_path))
tc  = cfg.get("text_config", cfg)
lt  = collections.Counter(tc.get("layer_types", []))

print("layer_types            :", dict(lt))
print("full_attention_interval:", tc.get("full_attention_interval"))
print()
print(f"linear_attention layers == Scan count : {lt['linear_attention'] == ops['Scan']}")
print(f"full_attention layers   == GQA count  : {lt['full_attention']  == ops['GroupQueryAttention']}")

## 5. How the work scales with prompt length

Sequential body executions before the first token = `prompt_tokens x Scan_layers x body_nodes`.
These are *graph* quantities, not measurements — but they show the shape of the cost:
work grows linearly in prompt length and none of it is batched across positions.

In [ ]:
n_scan, n_body = ops["Scan"], len(body.node)
print(f"{'prompt tokens':>14} | {'sequential body-node executions':>32}")
print("-" * 50)
for t in (60, 252, 1024, 2044):
    print(f"{t:>14} | {t * n_scan * n_body:>32,}")
print(f"\n(= tokens x {n_scan} Scan layers x {n_body} nodes per body)")

## Summary

| finding | value |
|---|---|
| `Scan` ops (sequential recurrence) | **24** — one per `linear_attention` layer |
| `GroupQueryAttention` | 8 |
| `LinearAttention` (ORT fused contrib op) | **0** — this graph never uses it |
| Where the `Scan`s sit | `else_branch` of `If(/model/layers.N/gdn/is_decode)` → **prefill** |
| Scan body | 16 nodes, the gated delta rule |
| `scan_input_axes` | `[2,2,2,2,2]` → walks the sequence one position at a time |

Two consequences worth noting:

1. Since `LinearAttention` never appears, changes to that ORT kernel cannot affect this model —
   24 of its 32 layers are not attention at all. That is a plausible reason
   [microsoft/onnxruntime#27780](https://github.com/microsoft/onnxruntime/pull/27780) did not move the number.
2. The gated delta rule is *chunkable* — reference implementations evaluate a block of positions
   together and carry the state across chunks. If the export only emits the sequential form, this
   is a **prefill** issue arising in the ONNX export rather than in the runtime's shaders.

**Open question for maintainers:** is the sequential `Scan` intentional for this export, or is there
a chunked form that could be emitted instead?

---

# Appendix — optional: does the runtime actually *execute* it that way?

Everything above is the graph as published. The fair objection is that a runtime might
optimize the `Scan` away at session-init, so the shipped graph wouldn't prove anything about
execution. This section checks that directly with ONNX Runtime's profiler.

**This part is heavier:** it downloads the real weights (~2.5 GB) and runs prefill on CPU.
A free CPU runtime is enough; it takes a few minutes. Everything above still stands on its
own if you skip this.

**Read the timings as CPU timings.** transformers.js runs WebGPU in a browser, which this
notebook cannot be. What *does* carry over is the **invocation count** — how many times each
op is executed is a property of graph execution, not of the backend. On a GPU backend those
invocations are kernel dispatches.

In [ ]:
%pip install -q onnxruntime numpy

In [ ]:
from huggingface_hub import hf_hub_download

# The graph references these by name as external data; they must sit next to it.
for f in ["onnx/decoder_model_merged_q4f16.onnx",
          "onnx/decoder_model_merged_q4f16.onnx_data",
          "onnx/decoder_model_merged_q4f16.onnx_data_1"]:
    local = hf_hub_download(REPO, f)
print("weights ready:", local)

In [ ]:
import onnxruntime as ort, numpy as np, json, os, time, collections, resource

MODEL = hf_hub_download(REPO, "onnx/decoder_model_merged_q4f16.onnx")
BODY  = {"Mul", "ReduceSum", "Exp", "Sub", "Add", "Unsqueeze"}   # the 16-node delta rule
NPT   = {"tensor(float16)": np.float16, "tensor(float)": np.float32,
         "tensor(int64)": np.int64, "tensor(int32)": np.int32, "tensor(bool)": np.bool_}

def prefill(seq_len):
    so = ort.SessionOptions()
    so.enable_profiling = True
    so.profile_file_prefix = f"p{seq_len}"
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL  # full opt
    sess = ort.InferenceSession(MODEL, so, providers=["CPUExecutionProvider"])

    dims, feed = {"batch_size": 1, "sequence_length": seq_len,
                  "past_sequence_length": 0, "total_sequence_length": seq_len}, {}
    for i in sess.get_inputs():
        shape = [dims.get(d, 1) if isinstance(d, str) else d for d in i.shape]
        dt = NPT[i.type]
        if i.name == "position_ids":       # rank-3: mRoPE sections
            feed[i.name] = np.broadcast_to(np.arange(seq_len, dtype=dt), tuple(shape)).astype(dt).copy()
        elif i.name == "attention_mask":   feed[i.name] = np.ones(shape, dtype=dt)
        elif i.name == "inputs_embeds":    feed[i.name] = (np.random.randn(*shape) * 0.02).astype(dt)
        else:                              feed[i.name] = np.zeros(shape, dtype=dt)

    # Exactly one run per session: the profiler records every execution, so a
    # warm-up pass would double every count reported below.
    t0 = time.time(); sess.run(["logits"], feed); wall = time.time() - t0
    pf = sess.end_profiling(); events = json.load(open(pf)); os.remove(pf)

    cnt, dur = collections.Counter(), collections.defaultdict(float)
    for e in events:
        if e.get("cat") == "Node" and e.get("dur") is not None:
            op = e.get("args", {}).get("op_name")
            if op:
                cnt[op] += 1
                dur[op] += e["dur"] / 1e6
    return wall, cnt, dur

rows = []
for n in (16, 32, 64, 128):
    wall, cnt, dur = prefill(n)
    rows.append((n, wall, cnt["Scan"], sum(cnt[o] for o in BODY), dur["Scan"], dur["MatMulNBits"]))
    print(f"seq={n:<4} wall={wall:6.2f}s  Scan nodes={cnt['Scan']}  "
          f"body-op invocations={sum(cnt[o] for o in BODY):,}")

peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
print(f"\npeak RSS: {peak / (1024**3 if os.uname().sysname=='Darwin' else 1024**2):.1f} GB")

### Did the optimizer remove the `Scan`?

`Scan` node count stays at 24 under `ORT_ENABLE_ALL`. The recurrence survives full graph
optimization — it is executed, not folded away.

### How does the work scale?

In [ ]:
n_scan, n_body = ops["Scan"], len(body.node)
per_token = n_scan * n_body

print(f"predicted invocations = {per_token} x seq + C     ({n_scan} Scan layers x {n_body} body nodes)\n")
print(f"{'seq':>5}{'observed':>11}{'predicted':>11}{'residual':>10}{'Scan s':>9}{'MatMulNBits s':>15}")
print("-" * 61)
resid = []
for seq, wall, nscan, body_n, d_scan, d_mm in rows:
    r = body_n - per_token * seq
    resid.append(r)
    print(f"{seq:>5}{body_n:>11,}{per_token*seq:>11,}{r:>10}{d_scan:>9.2f}{d_mm:>15.2f}")
print("-" * 61)
print(f"residual identical at every length: {len(set(resid)) == 1}  -> {resid}")
print("(a constant residual = ops of those types used elsewhere in the graph;")
print(" everything that grows is inside the Scan bodies)\n")

f = rows[-1][0] // rows[0][0]
print(f"observed wall-time growth over a {f}x longer prompt (linear would be x{f:.2f}):")
print(f"  Scan        x{rows[-1][4]/rows[0][4]:.2f}")
print(f"  MatMulNBits x{rows[-1][5]/rows[0][5]:.2f}")
print("  NOTE: these are single, un-warmed runs on a possibly shared machine, so the")
print("  ratios move between runs. The invocation relation above does not - it is exact.")
print(f"\nextrapolated to a 1024-token prompt: {per_token*1024 + resid[0]:,} sequential body-op invocations")

### What this establishes

**The runtime really does execute the recurrence position-by-position.** Under full graph
optimization the 24 `Scan` nodes survive — the recurrence is not folded away — and the number
of body-op invocations obeys `384 x prompt_tokens + C` exactly, with an identical residual at
every length across an 8x range. That is a counting result, not a fitted trend, and it comes
out byte-identical on repeated runs and different machines.

**Why that matters for a GPU backend.** Invocation counts are a property of graph execution,
not of the backend, so this count is what a WebGPU run would face too — where each invocation
is a kernel dispatch rather than a cheap CPU function call. A 1024-token prompt implies ~394k
of them, serialized, before the first token can appear.

The wall-clock columns are included for completeness, but treat them as indicative only: they
move noticeably with runtime load, and this notebook is likely sharing a machine. The counting
result is the part that reproduces.